In [1]:
import numpy as np
import pandas as pd
import warnings
import scipy.stats as stats

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

try:
    from skrvm import RVR 
except ImportError:
    RVR = None
    print("Uyarı: skrvm (RVM) bulunamadı.")

try:
    from ngboost import NGBRegressor
    from ngboost.distns import Laplace, LogNormal
except ImportError:
    NGBRegressor = None
    print("Uyarı: ngboost bulunamadı.")

node_available = False
try:
    import torch
    from pytorch_tabular import TabularModel
    from pytorch_tabular.models import NodeConfig
    node_available = True

except ImportError:
    print("Uyarı: pytorch_tabular veya torch bulunamadı. Node modelini kullanamayacaksınız.")

warnings.filterwarnings('ignore')

In [2]:
X_train = pd.read_csv('../../data/processed/X_train_v2.csv')
y_train_df = pd.read_csv('../../data/processed/y_train_v2.csv')

y_train_tweedie = y_train_df.iloc[:, 0].values

best_p = 1.543
power = 2 - best_p

def inverse_transform(y_pred):
    y_pred_safe = np.maximum(y_pred, 0)
    return np.power(y_pred_safe * power, 1 / power)

def forward_transform(y):
    return np.power(y, power) / power

y_train_raw = inverse_transform(y_train_tweedie)

print(f"Tweedie Hedef Max: {np.max(y_train_tweedie):.2f}")
print(f"Ham Hedef (RAW) Max: {np.max(y_train_raw):.2f} Hektar")

Tweedie Hedef Max: 44.98
Ham Hedef (RAW) Max: 746.28 Hektar


In [3]:
results_df = pd.DataFrame(columns=[
    'Loss_Function', 'Data_Type', 'Model', 
    'MedAE', 'MAE', 'MAD', 'RMSE', 'Spearman', 'Zero_HA_Acc', 
    'Coverage_90%', 'Avg_Interval_Width'
])

def evaluate_and_append(loss_name, data_type, model_name, y_true_raw, y_pred_raw, coverage, avg_width):
    global results_df
    y_pred_safe = np.maximum(y_pred_raw, 0)
    
    medae = median_absolute_error(y_true_raw, y_pred_safe)
    mae = mean_absolute_error(y_true_raw, y_pred_safe)
    mad = np.median(np.abs(y_pred_safe - np.median(y_pred_safe)))
    rmse = np.sqrt(mean_squared_error(y_true_raw, y_pred_safe))
    
    spearman_corr, _ = stats.spearmanr(y_true_raw, y_pred_safe)
    spearman_corr = 0.0 if np.isnan(spearman_corr) else spearman_corr
        
    zero_mask = (y_true_raw == 0)
    
    zero_ha_acc = np.mean(y_pred_safe[zero_mask] <= 0.5) * 100 if np.sum(zero_mask) > 0 else 0.0
        
    new_row = {
        'Loss_Function': loss_name,
        'Data_Type': data_type,
        'Model': model_name,
        'MedAE': round(medae, 4),
        'MAE': round(mae, 4),
        'MAD': round(mad, 4),
        'RMSE': round(rmse, 4),
        'Spearman': round(spearman_corr, 4),
        'Zero_HA_Acc': round(zero_ha_acc, 2),
        'Coverage_90%': round(coverage, 2),
        'Avg_Interval_Width': round(avg_width, 2)
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    print(f"  -> {data_type} tamamlandı. (MedAE: {medae:.2f})")

def conformal_cv(model, X, y_target, y_true_raw, is_tweedie, alpha=0.10):
    oof_preds_raw = np.zeros(len(X))
    coverage_flags = np.zeros(len(X))
    interval_widths = np.zeros(len(X))
    
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_target[train_idx], y_target[val_idx]
        
        model.fit(X_tr, y_tr)
        preds_val = model.predict(X_val)
        preds_tr = model.predict(X_tr)
        
        residuals_tr = np.abs(y_tr - preds_tr)
        margin = np.quantile(residuals_tr, 1 - alpha)
        
        lower_bound = preds_val - margin
        upper_bound = preds_val + margin
        
        if is_tweedie:
            preds_val_raw = inverse_transform(preds_val)
            lower_raw = inverse_transform(lower_bound)
            upper_raw = inverse_transform(upper_bound)
        else:
            preds_val_raw = np.maximum(preds_val, 0)
            lower_raw = np.maximum(lower_bound, 0)
            upper_raw = np.maximum(upper_bound, 0)
            
        oof_preds_raw[val_idx] = preds_val_raw
        interval_widths[val_idx] = upper_raw - lower_raw
        
        y_true_val = y_true_raw[val_idx]
        coverage_flags[val_idx] = (y_true_val >= lower_raw) & (y_true_val <= upper_raw)
        
    return oof_preds_raw, np.mean(coverage_flags) * 100, np.mean(interval_widths)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("Karar Mekanizması Hazır (1 Hektar Tolerans).")

Karar Mekanizması Hazır (1 Hektar Tolerans).


In [4]:
import logging
import os
import sys
import pandas as pd
from contextlib import contextmanager
from sklearn.base import BaseEstimator, RegressorMixin
from pytorch_tabular import TabularModel
from pytorch_tabular.models import NodeConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig

@contextmanager
def suppress_output():
    logger = logging.getLogger("pytorch_tabular")
    pl_logger = logging.getLogger("lightning.pytorch")
    
    old_level = logger.level
    pl_old_level = pl_logger.level
    
    logger.setLevel(logging.WARNING)
    pl_logger.setLevel(logging.WARNING)
    
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:  
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr
            logger.setLevel(old_level)
            pl_logger.setLevel(pl_old_level)

class SklearnNODE(BaseEstimator, RegressorMixin):
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.model = None
        self._is_first_fit = True

    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        
        trainer_config = TrainerConfig(
            batch_size=32, 
            max_epochs=20, 
            early_stopping=None, 
            progress_bar="none",
            trainer_kwargs={"enable_model_summary": False}
        )
        
        optimizer_config = OptimizerConfig()
        model_config = NodeConfig(task="regression", num_layers=2, num_trees=1024, depth=4, metrics=["mean_squared_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        with suppress_output():
            self.model.fit(train_df)
            
        if self._is_first_fit:
            print("  -> [NODE] Kurulum Başarılı (17.1M Parametre).")
            self._is_first_fit = False
            
        return self

    def predict(self, X):
        test_df = pd.DataFrame(X)
        test_df.columns = [f"col_{i}" for i in range(test_df.shape[1])]
        with suppress_output():
            preds_df = self.model.predict(test_df)
        pred_col = [col for col in preds_df.columns if 'prediction' in col.lower()][0]
        return preds_df[pred_col].values

## MSE LOSS İncelemesi:

In [5]:
from ngboost.distns import Normal
from IPython.display import display

results_df = results_df.iloc[0:0] 

models_mse = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "RVM": RVR(kernel='rbf'),
    "XGBoost": XGBRegressor(objective='reg:squarederror', random_state=42),
    "RF": RandomForestRegressor(criterion='squared_error', random_state=42),
    "NODE": SklearnNODE(random_state=42),
    "NGBoost": NGBRegressor(Dist=Normal, random_state=42, verbose=False)
}

print("--- EĞİTİM VE TEST BAŞLIYOR (MSE LOSS) ---\n")

for name, model in models_mse.items():
    print(f"[{name}] eğitiliyor...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('MSE', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('MSE', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== MSE & CONFORMAL SONUÇLARI ==================\n")
mse_results = results_df[results_df['Loss_Function'] == 'MSE'].copy()

styled_df = mse_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_df)

--- EĞİTİM VE TEST BAŞLIYOR (MSE LOSS) ---

[SVM] eğitiliyor...
  -> Tweedie tamamlandı. (MedAE: 1.35)
  -> Raw tamamlandı. (MedAE: 1.41)
--------------------------------------------------
[RVM] eğitiliyor...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor...
  -> Tweedie tamamlandı. (MedAE: 3.22)
  -> Raw tamamlandı. (MedAE: 5.78)
--------------------------------------------------
[RF] eğitiliyor...
  -> Tweedie tamamlandı. (MedAE: 3.41)


2026-07-28 14:02:10,351 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 8.02)
--------------------------------------------------
[NODE] eğitiliyor...
  -> [NODE] Kurulum Başarılı (17.1M Parametre).


2026-07-28 14:06:25,330 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:10:32,459 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:14:39,485 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:18:38,661 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:23:19,444 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.24)


2026-07-28 14:27:28,937 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:31:39,130 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:35:53,253 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:40:06,078 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 1.01)
--------------------------------------------------
[NGBoost] eğitiliyor...
  -> Tweedie tamamlandı. (MedAE: 2.76)
  -> Raw tamamlandı. (MedAE: 6.09)
--------------------------------------------------

================== MSE & CONFORMAL SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
0,MSE,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
1,MSE,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92
2,MSE,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
3,MSE,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
4,MSE,Tweedie,XGBoost,3.22,15.91,1.40,56.22,0.03,33.33,4.36,0.11
5,MSE,Raw,XGBoost,5.78,22.40,3.98,78.58,0.06,20.71,8.72,0.15
6,MSE,Tweedie,RF,3.41,13.42,1.44,47.87,0.06,13.13,50.36,11.93
7,MSE,Raw,RF,8.02,20.16,4.19,54.49,0.03,0.00,72.15,24.63
8,MSE,Tweedie,NODE,0.24,11.06,0.00,46.98,-0.04,100.00,89.83,23.70
9,MSE,Raw,NODE,1.01,11.07,0.01,46.80,0.04,0.00,90.07,23.69


## MAE LOSS İncelemesi:

In [6]:
from ngboost.distns import Laplace
from IPython.display import display
import torch.nn as nn

class SklearnNODE_MAE(SklearnNODE):
    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        trainer_config = TrainerConfig(batch_size=32, max_epochs=20, early_stopping=None, progress_bar="none", trainer_kwargs={"enable_model_summary": False})
        optimizer_config = OptimizerConfig()
        
        model_config = NodeConfig(task="regression", loss="L1Loss", num_layers=2, num_trees=1024, depth=4, metrics=["mean_absolute_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        with suppress_output():
            self.model.fit(train_df)
            
        if getattr(self, '_is_first_fit', True):
            print("  -> [NODE] MAE Loss (L1) ile Başarıyla Kuruldu (17.1M Parametre).")
            self._is_first_fit = False
        return self

models_mae = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.0), 
    "RVM": RVR(kernel='rbf'), 
    "XGBoost": XGBRegressor(objective='reg:absoluteerror', random_state=42),
    "RF": RandomForestRegressor(criterion='absolute_error', random_state=42),
    "NODE": SklearnNODE_MAE(random_state=42),
    "NGBoost": NGBRegressor(Dist=Laplace, random_state=42, verbose=False) # Laplace dağılımı = MAE hedeflidir
}

for name, model in models_mae.items():
    print(f"[{name}] eğitiliyor (Kayıp Fonksiyonu: MAE)...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('MAE', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('MAE', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== MAE LOSS & CONFORMAL SONUÇLARI ==================\n")
mae_results = results_df[results_df['Loss_Function'] == 'MAE'].copy()

styled_mae = mae_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_mae)

[SVM] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> Tweedie tamamlandı. (MedAE: 1.38)
  -> Raw tamamlandı. (MedAE: 1.35)
--------------------------------------------------
[RVM] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> Tweedie tamamlandı. (MedAE: 3.17)
  -> Raw tamamlandı. (MedAE: 5.10)
--------------------------------------------------
[RF] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> Tweedie tamamlandı. (MedAE: 3.10)


2026-07-28 14:50:33,400 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 7.37)
--------------------------------------------------
[NODE] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> [NODE] MAE Loss (L1) ile Başarıyla Kuruldu (17.1M Parametre).


2026-07-28 14:55:11,032 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 14:59:10,489 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:03:15,049 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:07:18,820 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:11:17,175 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.42)


2026-07-28 15:15:19,440 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:19:28,253 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:23:30,858 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:27:38,816 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 0.87)
--------------------------------------------------
[NGBoost] eğitiliyor (Kayıp Fonksiyonu: MAE)...
  -> Tweedie tamamlandı. (MedAE: 1.41)
  -> Raw tamamlandı. (MedAE: 2.25)
--------------------------------------------------

================== MAE LOSS & CONFORMAL SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
12,MAE,Tweedie,SVM,1.38,11.13,0.26,46.91,-0.07,52.02,89.59,23.29
13,MAE,Raw,SVM,1.35,11.13,0.20,46.89,-0.09,47.47,90.07,23.92
14,MAE,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
15,MAE,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
16,MAE,Tweedie,XGBoost,3.17,12.99,1.43,47.20,0.04,39.39,34.14,5.28
17,MAE,Raw,XGBoost,5.10,18.70,3.31,58.21,0.01,19.19,51.33,9.06
18,MAE,Tweedie,RF,3.10,12.45,1.29,46.86,0.02,8.08,51.82,11.79
19,MAE,Raw,RF,7.37,17.93,3.79,50.13,-0.01,0.51,74.09,23.29
20,MAE,Tweedie,NODE,0.42,11.07,0.01,47.02,-0.07,100.00,90.07,23.68
21,MAE,Raw,NODE,0.87,11.09,0.10,46.94,-0.08,74.24,89.83,23.69


## HUBER LOSS İncelemesi:

In [7]:
class SklearnNODE_Huber(SklearnNODE):
    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        trainer_config = TrainerConfig(batch_size=32, max_epochs=20, early_stopping=None, progress_bar="none", trainer_kwargs={"enable_model_summary": False})
        optimizer_config = OptimizerConfig()
        
        model_config = NodeConfig(task="regression", loss="HuberLoss", num_layers=2, num_trees=1024, depth=4, metrics=["mean_absolute_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        with suppress_output():
            self.model.fit(train_df)
            
        if getattr(self, '_is_first_fit', True):
            print("  -> [NODE] Huber Loss ile Kuruldu (17.1M Parametre).")
            self._is_first_fit = False
        return self


models_huber = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "RVM": RVR(kernel='rbf') if 'RVR' in globals() else None, 
    "XGBoost": XGBRegressor(objective='reg:pseudohubererror', random_state=42),
    "RF": RandomForestRegressor(criterion='squared_error', random_state=42),
    "NODE": SklearnNODE_Huber(random_state=42),
    "NGBoost": NGBRegressor(Dist=Normal, random_state=42, verbose=False) 
}
models_huber = {k: v for k, v in models_huber.items() if v is not None}

for name, model in models_huber.items():
    print(f"[{name}] eğitiliyor (Kayıp Fonksiyonu: Huber)...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('Huber', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('Huber', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== HUBER LOSS & CONFORMAL SONUÇLARI ==================\n")
huber_results = results_df[results_df['Loss_Function'] == 'Huber'].copy()

styled_huber = huber_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_huber)

[SVM] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> Tweedie tamamlandı. (MedAE: 1.35)
  -> Raw tamamlandı. (MedAE: 1.41)
--------------------------------------------------
[RVM] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> Tweedie tamamlandı. (MedAE: 3.75)
  -> Raw tamamlandı. (MedAE: 4.09)
--------------------------------------------------
[RF] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> Tweedie tamamlandı. (MedAE: 3.41)


2026-07-28 15:35:48,733 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 8.02)
--------------------------------------------------
[NODE] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> [NODE] Huber Loss ile Kuruldu (17.1M Parametre).


2026-07-28 15:40:02,846 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:44:04,244 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:48:13,091 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 15:52:34,848 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.31)


2026-07-28 15:57:29,161 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 16:03:12,758 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 16:08:57,693 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 16:14:08,506 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 16:20:19,287 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 0.90)
--------------------------------------------------
[NGBoost] eğitiliyor (Kayıp Fonksiyonu: Huber)...
  -> Tweedie tamamlandı. (MedAE: 2.76)
  -> Raw tamamlandı. (MedAE: 6.09)
--------------------------------------------------

================== HUBER LOSS & CONFORMAL SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
24,Huber,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
25,Huber,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92
26,Huber,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
27,Huber,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
28,Huber,Tweedie,XGBoost,3.75,13.27,1.79,47.16,-0.05,29.29,10.41,1.11
29,Huber,Raw,XGBoost,4.09,12.53,2.50,46.58,-0.06,18.18,70.22,11.45
30,Huber,Tweedie,RF,3.41,13.42,1.44,47.87,0.06,13.13,50.36,11.93
31,Huber,Raw,RF,8.02,20.16,4.19,54.49,0.03,0.00,72.15,24.63
32,Huber,Tweedie,NODE,0.31,11.07,0.02,47.00,-0.14,100.00,90.07,23.74
33,Huber,Raw,NODE,0.90,11.07,0.05,46.85,-0.14,0.00,89.83,23.68


## LOG-COSH LOSS İncelemesi:

In [9]:
print("--- HÜCRE 7: YALNIZCA LOG-COSH LOSS İNCELEMESİ ---\n")

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display

def log_cosh_obj(y_true, y_pred):
    x = y_pred - y_true
    grad = np.tanh(x)                     
    hess = 1.0 - grad**2                  
    hess = np.maximum(hess, 1e-6)         
    return grad, hess

class LogCoshLossPyTorch(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, y_pred, y_true):
        x = y_pred - y_true
        abs_x = torch.abs(x)
        loss = abs_x - np.log(2.0) + torch.nn.functional.softplus(-2.0 * abs_x)
        return torch.mean(loss)

class SklearnNODE_LogCosh(SklearnNODE):
    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        trainer_config = TrainerConfig(batch_size=32, max_epochs=20, early_stopping=None, progress_bar="none", trainer_kwargs={"enable_model_summary": False})
        optimizer_config = OptimizerConfig()
        
        model_config = NodeConfig(task="regression", loss="MSELoss", num_layers=2, num_trees=1024, depth=4, metrics=["mean_absolute_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        custom_loss = LogCoshLossPyTorch()
        with suppress_output():
            self.model.fit(train_df, loss=custom_loss)
            
        if getattr(self, '_is_first_fit', True):
            print("  -> [NODE] Custom Log-Cosh Loss ile Kuruldu (17.1M Parametre).")
            self._is_first_fit = False
        return self

models_logcosh = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "RVM": RVR(kernel='rbf') if 'RVR' in globals() else None, 
    "XGBoost": XGBRegressor(objective=log_cosh_obj, random_state=42),
    "RF": RandomForestRegressor(criterion='squared_error', random_state=42),
    "NODE": SklearnNODE_LogCosh(random_state=42),
    "NGBoost": NGBRegressor(Dist=Normal, random_state=42, verbose=False) 
}

models_logcosh = {k: v for k, v in models_logcosh.items() if v is not None}

for name, model in models_logcosh.items():
    print(f"[{name}] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('Log-Cosh', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('Log-Cosh', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== LOG-COSH LOSS & CONFORMAL SONUÇLARI ==================\n")
logcosh_results = results_df[results_df['Loss_Function'] == 'Log-Cosh'].copy()

styled_logcosh = logcosh_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_logcosh)

--- HÜCRE 7: YALNIZCA LOG-COSH LOSS İNCELEMESİ ---

[SVM] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> Tweedie tamamlandı. (MedAE: 1.35)
  -> Raw tamamlandı. (MedAE: 1.41)
--------------------------------------------------
[RVM] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> Tweedie tamamlandı. (MedAE: 3.22)
  -> Raw tamamlandı. (MedAE: 3.98)
--------------------------------------------------
[RF] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> Tweedie tamamlandı. (MedAE: 3.41)


2026-07-28 21:27:28,304 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 8.02)
--------------------------------------------------
[NODE] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> [NODE] Custom Log-Cosh Loss ile Kuruldu (17.1M Parametre).


2026-07-28 21:34:27,245 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 21:41:04,674 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 21:46:47,063 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 21:52:17,242 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 21:56:42,656 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.29)


2026-07-28 22:02:11,051 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:07:55,307 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:13:27,042 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:18:57,344 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 0.96)
--------------------------------------------------
[NGBoost] eğitiliyor (Kayıp Fonksiyonu: Log-Cosh)...
  -> Tweedie tamamlandı. (MedAE: 2.76)
  -> Raw tamamlandı. (MedAE: 6.09)
--------------------------------------------------

================== LOG-COSH LOSS & CONFORMAL SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
36,Log-Cosh,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
37,Log-Cosh,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92
38,Log-Cosh,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
39,Log-Cosh,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
40,Log-Cosh,Tweedie,XGBoost,3.22,13.31,1.79,47.23,-0.03,31.31,9.69,1.11
41,Log-Cosh,Raw,XGBoost,3.98,12.45,2.40,46.53,-0.03,16.16,71.43,11.53
42,Log-Cosh,Tweedie,RF,3.41,13.42,1.44,47.87,0.06,13.13,50.36,11.93
43,Log-Cosh,Raw,RF,8.02,20.16,4.19,54.49,0.03,0.00,72.15,24.63
44,Log-Cosh,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
45,Log-Cosh,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92


## TWEEDİE LOSS İncelemesi:

In [10]:
import torch
import torch.nn as nn
from IPython.display import display

results_df = results_df.drop_duplicates(subset=['Loss_Function', 'Data_Type', 'Model'], keep='last').reset_index(drop=True)

class TweedieLossPyTorch(nn.Module):
    def __init__(self, p=1.5):
        super().__init__()
        self.p = p # Variance Power (1.5 = Poisson ve Gamma karışımı)

    def forward(self, y_pred, y_true):
        y_pred_pos = torch.nn.functional.softplus(y_pred) + 1e-6
        
        term1 = - y_true * torch.pow(y_pred_pos, 1 - self.p) / (1 - self.p)
        term2 = torch.pow(y_pred_pos, 2 - self.p) / (2 - self.p)
        
        return torch.mean(term1 + term2)

class SklearnNODE_Tweedie(SklearnNODE):
    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        trainer_config = TrainerConfig(batch_size=32, max_epochs=20, early_stopping=None, progress_bar="none", trainer_kwargs={"enable_model_summary": False})
        optimizer_config = OptimizerConfig()
        
        model_config = NodeConfig(task="regression", loss="MSELoss", num_layers=2, num_trees=1024, depth=4, metrics=["mean_absolute_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        custom_tweedie = TweedieLossPyTorch(p=1.5)
        with suppress_output():
            self.model.fit(train_df, loss=custom_tweedie)
            
        if getattr(self, '_is_first_fit', True):
            print("  -> [NODE] Custom Tweedie Loss ile Kuruldu (17.1M Parametre).")
            self._is_first_fit = False
        return self

    def predict(self, X):
        test_df = pd.DataFrame(X)
        test_df.columns = [f"col_{i}" for i in range(test_df.shape[1])]
        with suppress_output():
            preds_df = self.model.predict(test_df)
        pred_col = [col for col in preds_df.columns if 'prediction' in col.lower()][0]
        raw_preds = preds_df[pred_col].values
        
        return np.log1p(np.exp(np.clip(raw_preds, -10, 10)))

models_tweedie = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "RVM": RVR(kernel='rbf') if 'RVR' in globals() else None, 
    "XGBoost": XGBRegressor(objective='reg:tweedie', tweedie_variance_power=1.5, random_state=42),
    "RF": RandomForestRegressor(criterion='squared_error', random_state=42),
    "NODE": SklearnNODE_Tweedie(random_state=42),
    "NGBoost": NGBRegressor(Dist=Normal, random_state=42, verbose=False) 
}

models_tweedie = {k: v for k, v in models_tweedie.items() if v is not None}

for name, model in models_tweedie.items():
    print(f"[{name}] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('Tweedie_Obj', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('Tweedie_Obj', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== TWEEDIE OBJECTIVE SONUÇLARI ==================\n")
tweedie_results = results_df[results_df['Loss_Function'] == 'Tweedie_Obj'].copy()

styled_tweedie = tweedie_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_tweedie)

[SVM] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> Tweedie tamamlandı. (MedAE: 1.35)
  -> Raw tamamlandı. (MedAE: 1.41)
--------------------------------------------------
[RVM] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> Tweedie tamamlandı. (MedAE: 1.49)
  -> Raw tamamlandı. (MedAE: 1.83)
--------------------------------------------------
[RF] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> Tweedie tamamlandı. (MedAE: 3.41)


2026-07-28 22:35:54,016 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 8.02)
--------------------------------------------------
[NODE] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> [NODE] Custom Tweedie Loss ile Kuruldu (17.1M Parametre).


2026-07-28 22:39:58,621 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:44:10,296 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:48:21,455 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 22:52:31,008 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:01:29,333 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.34)


2026-07-28 23:05:40,196 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:09:49,901 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:13:49,805 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:17:51,498 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 1.31)
--------------------------------------------------
[NGBoost] eğitiliyor (Kayıp Fonksiyonu: Tweedie Objective)...
  -> Tweedie tamamlandı. (MedAE: 2.76)
  -> Raw tamamlandı. (MedAE: 6.09)
--------------------------------------------------

================== TWEEDIE OBJECTIVE SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
48,Tweedie_Obj,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
49,Tweedie_Obj,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92
50,Tweedie_Obj,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
51,Tweedie_Obj,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
52,Tweedie_Obj,Tweedie,XGBoost,1.49,12.19,0.01,47.92,-0.01,79.80,28.57,0.53
53,Tweedie_Obj,Raw,XGBoost,1.83,12.30,0.30,47.76,-0.00,56.06,34.14,1.07
54,Tweedie_Obj,Tweedie,RF,3.41,13.42,1.44,47.87,0.06,13.13,50.36,11.93
55,Tweedie_Obj,Raw,RF,8.02,20.16,4.19,54.49,0.03,0.00,72.15,24.63
56,Tweedie_Obj,Tweedie,NODE,0.34,11.06,0.00,46.95,-0.08,100.00,89.83,23.69
57,Tweedie_Obj,Raw,NODE,1.31,11.10,0.01,46.74,0.03,0.00,90.07,23.69


## ASİMETRİK LOSS İncelemesi:

In [11]:
results_df = results_df.drop_duplicates(subset=['Loss_Function', 'Data_Type', 'Model'], keep='last').reset_index(drop=True)

def asymmetric_mse_obj(y_true, y_pred):
    x = y_pred - y_true
    penalty = 3.0 # Underpredict (Eksik Tahmin) için 3 Kat Ceza
    
    grad = np.where(x < 0, penalty * 2.0 * x, 2.0 * x)
    hess = np.where(x < 0, penalty * 2.0, 2.0)
    return grad, hess

class AsymmetricMSELossPyTorch(nn.Module):
    def __init__(self, penalty=3.0):
        super().__init__()
        self.penalty = penalty

    def forward(self, y_pred, y_true):
        x = y_pred - y_true
        loss = torch.where(x < 0, self.penalty * (x ** 2), x ** 2)
        return torch.mean(loss)

class SklearnNODE_Asymmetric(SklearnNODE):
    def fit(self, X, y):
        train_df = pd.DataFrame(X)
        train_df.columns = [f"col_{i}" for i in range(train_df.shape[1])]
        train_df['target'] = y
        
        data_config = DataConfig(target=['target'], continuous_cols=list(train_df.columns[:-1]), categorical_cols=[])
        trainer_config = TrainerConfig(batch_size=32, max_epochs=20, early_stopping=None, progress_bar="none", trainer_kwargs={"enable_model_summary": False})
        optimizer_config = OptimizerConfig()
        
        model_config = NodeConfig(task="regression", loss="MSELoss", num_layers=2, num_trees=1024, depth=4, metrics=["mean_absolute_error"])
        
        self.model = TabularModel(
            data_config=data_config, model_config=model_config, 
            optimizer_config=optimizer_config, trainer_config=trainer_config, 
            verbose=False, suppress_lightning_logger=True
        )
        
        custom_loss = AsymmetricMSELossPyTorch(penalty=3.0)
        with suppress_output():
            self.model.fit(train_df, loss=custom_loss)
            
        if getattr(self, '_is_first_fit', True):
            print("  -> [NODE] Custom Asymmetric Loss (3x Ceza) ile Kuruldu.")
            self._is_first_fit = False
        return self

models_asym = {
    "SVM": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "RVM": RVR(kernel='rbf') if 'RVR' in globals() else None, 
    "XGBoost": XGBRegressor(objective=asymmetric_mse_obj, random_state=42),
    "RF": RandomForestRegressor(criterion='squared_error', random_state=42),
    "NODE": SklearnNODE_Asymmetric(random_state=42),
    "NGBoost": NGBRegressor(Dist=Normal, random_state=42, verbose=False) 
}

models_asym = {k: v for k, v in models_asym.items() if v is not None}

for name, model in models_asym.items():
    print(f"[{name}] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...")
    
    oof_preds, cov_tweedie, width_tweedie = conformal_cv(model, X_train, y_train_tweedie, y_train_raw, is_tweedie=True)
    evaluate_and_append('Asymmetric', 'Tweedie', name, y_train_raw, oof_preds, cov_tweedie, width_tweedie)
    
    oof_raw, cov_raw, width_raw = conformal_cv(model, X_train, y_train_raw, y_train_raw, is_tweedie=False)
    evaluate_and_append('Asymmetric', 'Raw', name, y_train_raw, oof_raw, cov_raw, width_raw)
    
    print("-" * 50)

print("\n================== ASYMMETRIC LOSS SONUÇLARI ==================\n")
asym_results = results_df[results_df['Loss_Function'] == 'Asymmetric'].copy()

styled_asym = asym_results.style.background_gradient(
    subset=['MedAE', 'MAE', 'MAD', 'RMSE', 'Avg_Interval_Width'], cmap='Reds'
).background_gradient(
    subset=['Spearman', 'Zero_HA_Acc', 'Coverage_90%'], cmap='Greens'
).format(precision=2)

display(styled_asym)

[SVM] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...
  -> Tweedie tamamlandı. (MedAE: 1.35)
  -> Raw tamamlandı. (MedAE: 1.41)
--------------------------------------------------
[RVM] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...
  -> Tweedie tamamlandı. (MedAE: 1.27)
  -> Raw tamamlandı. (MedAE: 1.08)
--------------------------------------------------
[XGBoost] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...
  -> Tweedie tamamlandı. (MedAE: 3.11)
  -> Raw tamamlandı. (MedAE: 6.36)
--------------------------------------------------
[RF] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...
  -> Tweedie tamamlandı. (MedAE: 3.41)
  -> Raw tamamlandı. (MedAE: 8.02)
--------------------------------------------------
[NODE] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...


2026-07-28 23:31:23,009 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> [NODE] Custom Asymmetric Loss (3x Ceza) ile Kuruldu.


2026-07-28 23:37:23,050 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:43:13,227 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:48:49,973 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 23:54:30,012 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-29 00:00:15,528 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Tweedie tamamlandı. (MedAE: 0.24)


2026-07-29 00:05:58,931 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-29 00:11:38,637 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-29 00:17:14,905 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-29 00:22:34,555 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....


  -> Raw tamamlandı. (MedAE: 1.01)
--------------------------------------------------
[NGBoost] eğitiliyor (Kayıp Fonksiyonu: Asymmetric 3x)...
  -> Tweedie tamamlandı. (MedAE: 2.76)
  -> Raw tamamlandı. (MedAE: 6.09)
--------------------------------------------------

================== ASYMMETRIC LOSS SONUÇLARI ==================



,Loss_Function,Data_Type,Model,MedAE,MAE,MAD,RMSE,Spearman,Zero_HA_Acc,Coverage_90%,Avg_Interval_Width
60,Asymmetric,Tweedie,SVM,1.35,11.12,0.22,46.91,-0.07,52.02,89.59,23.33
61,Asymmetric,Raw,SVM,1.41,11.13,0.17,46.89,-0.10,45.45,89.83,23.92
62,Asymmetric,Tweedie,RVM,1.27,11.69,0.01,47.21,-0.14,88.89,44.55,1.13
63,Asymmetric,Raw,RVM,1.08,12.19,0.09,47.42,-0.14,1.52,54.96,2.63
64,Asymmetric,Tweedie,XGBoost,3.11,14.52,1.40,49.36,0.03,38.89,7.26,0.07
65,Asymmetric,Raw,XGBoost,6.36,22.32,5.00,72.43,0.01,19.70,9.20,0.08
66,Asymmetric,Tweedie,RF,3.41,13.42,1.44,47.87,0.06,13.13,50.36,11.93
67,Asymmetric,Raw,RF,8.02,20.16,4.19,54.49,0.03,0.00,72.15,24.63
68,Asymmetric,Tweedie,NODE,0.24,11.06,0.00,46.98,-0.03,100.00,89.83,23.70
69,Asymmetric,Raw,NODE,1.01,11.07,0.01,46.80,0.05,0.00,90.07,23.69


In [ ]:
# .gitignore dosyasının sonuna istenmeyen klasörleri ekleyen güvenli Python kodu
with open(".gitignore", "a", encoding="utf-8") as f:
    f.write("\n# PyTorch Loglari\n")
    f.write("notebooks/Modeling_Phase2/lightning_logs/\n")
    f.write("notebooks/Modeling_Phase2/saved_models/\n")
    f.write("notebooks/Modeling_Phase2/.pt_tmp/\n")
    
print("İşlem Başarılı! .gitignore dosyası güncellendi.")